# Bài 13 — Multi-document RAG với Metadata Filtering

**Mục tiêu:** Mở rộng RAG từ 1 tài liệu → nhiều công ty, dùng metadata để tránh lẫn dữ liệu.

**Kết nối:** Phase 2 có RAG cho 1 công ty NVIDIA. Bài này thêm AMD vào cùng collection, dùng `where` filter để query đúng tài liệu.

---
## Nội dung
1. Lý thuyết — Data leakage & Metadata filtering
2. Ví dụ — Demo với dữ liệu giả (in-memory ChromaDB)
3. Bài tập — Ingest 2 báo cáo thực + viết `rag_agent(question, symbol)`

## 1. Lý thuyết

### Vấn đề: Data Leakage

Nếu thêm AMD vào cùng collection với NVIDIA **mà không đánh nhãn**:

```
Collection (không có metadata):
  chunk_0: "NVIDIA revenue $44B in Q1 2026"
  chunk_1: "AMD revenue $7.4B in Q1 2026"     ← chunk của AMD
  chunk_2: "NVIDIA data center grew 73%"
```

Hỏi: `"Doanh thu của công ty này là bao nhiêu?"` (ý định hỏi về NVIDIA)
→ Vector của *"revenue"* rất gần nhau → top kết quả có thể gồm cả `chunk_1` (AMD)
→ **Báo cáo NVIDIA bị lẫn số liệu AMD** — đây là data leakage.

---

### Giải pháp: Metadata Tagging + Where Filter

**Khi lưu chunk → gắn nhãn:**
```python
collection.add(
    documents=["NVIDIA revenue $44B in Q1 2026"],
    metadatas=[{"symbol": "NVDA"}],   # ← nhãn!
    ids=["NVDA_chunk_0"]
)
```

**Khi query → lọc theo nhãn:**
```python
collection.query(
    query_embeddings=[...],
    n_results=3,
    where={"symbol": "NVDA"}   # chỉ lấy chunk có nhãn NVDA
)
```

---

### Cú pháp `where` của ChromaDB

| Filter | Ý nghĩa |
|---|---|
| `{"symbol": "NVDA"}` | chỉ lấy chunks của NVDA |
| `{"symbol": {"$in": ["NVDA", "AMD"]}}` | lấy chunks của cả NVDA lẫn AMD |
| `{"year": {"$gte": "2025"}}` | lọc theo năm (ví dụ thêm metadata khác) |

Bài này ta dùng 2 filter đầu.

## 2. Ví dụ — Demo Metadata Isolation

Dùng ChromaDB **in-memory** và text giả để thấy rõ `where` filter hoạt động thế nào.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

embed_demo = SentenceTransformer("all-MiniLM-L6-v2")
demo_client = chromadb.Client()  # in-memory, không lưu file
demo_col = demo_client.create_collection("demo_metadata")

# Dữ liệu giả: 2 chunks NVDA + 1 chunk AMD
texts = [
    "NVIDIA reported revenue of 44 billion dollars in Q1 fiscal 2026",
    "NVIDIA data center revenue grew 73 percent year over year",
    "AMD reported revenue of 7.4 billion dollars in Q1 2026",
]
metadatas = [
    {"symbol": "NVDA"},
    {"symbol": "NVDA"},
    {"symbol": "AMD"},
]
embeddings = embed_demo.encode(texts).tolist()
demo_col.add(documents=texts, embeddings=embeddings, metadatas=metadatas, ids=["n1", "n2", "a1"])
print("Đã thêm 3 chunks (2 NVDA + 1 AMD)")

In [ ]:
query_vec = embed_demo.encode(["quarterly revenue financial results"]).tolist()

print("=== Query KHÔNG có filter ===")
r_all = demo_col.query(query_embeddings=query_vec, n_results=3)
for doc, meta in zip(r_all["documents"][0], r_all["metadatas"][0]):
    print(f"  [{meta['symbol']}] {doc[:70]}")

print()
print("=== Query CHỈ NVDA ===")
r_nvda = demo_col.query(query_embeddings=query_vec, n_results=3, where={"symbol": "NVDA"})
for doc, meta in zip(r_nvda["documents"][0], r_nvda["metadatas"][0]):
    print(f"  [{meta['symbol']}] {doc[:70]}")

print()
print("=== Query CHỈ AMD ===")
r_amd = demo_col.query(query_embeddings=query_vec, n_results=3, where={"symbol": "AMD"})
for doc, meta in zip(r_amd["documents"][0], r_amd["metadatas"][0]):
    print(f"  [{meta['symbol']}] {doc[:70]}")

**Quan sát:**
- Query **không filter**: top 3 kết quả gồm cả NVDA lẫn AMD (revenue gần nhau về vector)
- Query **where NVDA**: chỉ thấy 2 chunks NVDA, AMD bị loại hoàn toàn
- Query **where AMD**: chỉ thấy 1 chunk AMD

Đây chính là isolation ta cần cho multi-document RAG.

---
## 3. Bài tập

**File dùng trong bài — 4 file, 2 công ty:**
- NVDA 10-Q: `phase4/data/NVDA-F1Q26-10-Q.pdf`
- NVDA Presentation: `phase4/data/NVDA-F1Q26-Quarterly-Presentation-FINAL.pdf`
- AMD 10-Q: `phase4/data/May 6, 2026 - 10-Q...AMD.pdf`
- AMD Slides: `phase4/data/AMD Q1'26 Earnings Slides Final.pdf`

Cả 2 file NVDA đều có `symbol="NVDA"` → `where={"symbol": "NVDA"}` lấy từ cả 2 file cùng lúc. Đây là điểm mạnh của metadata approach: filter theo **công ty**, không phải theo tên file.

**Bạn cần làm 7 bước — mỗi ô code có chú thích TODO chỉ chỗ cần viết.**

In [18]:
# Setup — import và khởi tạo
import os
import fitz
import chromadb
from sentence_transformers import SentenceTransformer
from google import genai
from dotenv import load_dotenv

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# NVIDIA: 10-Q (báo cáo chi tiết) + Presentation (investor slides)
NVDA_10Q  = r"D:\Code\AI\Multi-Agent Research Assistant project\phase4\data\NVDA-F1Q26-10-Q.pdf"
NVDA_PRES = r"D:\Code\AI\Multi-Agent Research Assistant project\phase4\data\NVDA-F1Q26-Quarterly-Presentation-FINAL.pdf"

# AMD: 10-Q + Earnings Slides
AMD_10Q   = r"D:\Code\AI\Multi-Agent Research Assistant project\phase4\data\AMD_F1Q26-10-Q.pdf"
AMD_PRES  = r"D:\Code\AI\Multi-Agent Research Assistant project\phase4\data\AMD Q1'26 Earnings Slides Final.pdf"

CHROMA_PATH = r"D:\Code\AI\Multi-Agent Research Assistant project\phase4\chroma_db"

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11822.38it/s]


In [19]:
# Hàm chunking — tái sử dụng từ phase2, không cần thay đổi
def split_into_chunks(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

In [20]:
# Bước 1 — Khởi tạo ChromaDB collection
# Dùng collection KHÁC phase2 (tên: multi_company_reports)
db_client  = chromadb.PersistentClient(path=CHROMA_PATH)
collection = db_client.get_or_create_collection(name="multi_company_reports")
print(f"Collection '{collection.name}' | Hiện có: {collection.count()} chunks")

Collection 'multi_company_reports' | Hiện có: 58 chunks


In [21]:
# Bước 2 — chunk_pdf(pdf_path, symbol)
# Đọc PDF → tách chunks → gắn metadata {"symbol": symbol} vào từng chunk

def chunk_pdf(pdf_path, symbol):
    # Đọc toàn bộ text từ PDF (giống phase2)
    doc = fitz.open(pdf_path)
    full_text = ""
    for page_num in range(doc.page_count):
        page = doc[page_num]
        full_text += page.get_text()
    doc.close()

    chunks = split_into_chunks(full_text)

    # Lấy tên file (không có extension) để dùng trong ID
    doc_name = os.path.splitext(os.path.basename(pdf_path))[0][:20]  # cắt ngắn cho gọn

    # TODO: Tạo 3 list từ chunks, symbol và doc_name
    #
    # documents: chính là chunks (list các đoạn text)
    #
    # metadatas: list of dict, mỗi chunk cần 1 dict {"symbol": symbol}
    #            Đây là nhãn để where filter hoạt động sau này
    #
    # ids:       list string unique — QUAN TRỌNG: phải unique kể cả khi
    #            ingest nhiều file cùng symbol.
    #            Dùng format: f"{symbol}_{doc_name}_chunk_{i}"
    #            (có doc_name để NVDA_10Q và NVDA_PRES không bị trùng ID)
    documents = chunks
    metadatas = [{"symbol": symbol} for _ in chunks] 
    ids       = [
        f"{symbol}-{doc_name}_chunk_{i}" for i in range(len(chunks))
    ]

    print(f"[{symbol}] {len(documents)} chunks từ {os.path.basename(pdf_path)}")
    return documents, metadatas, ids

In [22]:
# Bước 3 — ingest_company(pdf_path, symbol, collection)
# Đóng gói toàn bộ pipeline: chunk → embed → lưu ChromaDB

def ingest_company(pdf_path, symbol, collection):
    documents, metadatas, ids = chunk_pdf(pdf_path, symbol)
    embeddings = embed_model.encode(documents).tolist()

    # TODO: gọi collection.add() với đủ 4 tham số:
    #        documents=, embeddings=, metadatas=, ids=
    #
    # Lưu ý: nếu chạy notebook lần 2 bị lỗi "ID already exists"
    #        → đổi collection.add() thành collection.upsert()
    collection.add(
        documents = documents,
        embeddings = embeddings,
        metadatas = metadatas,
        ids = ids
    )

    print(f"[{symbol}] Đã ingest {len(documents)} chunks vào ChromaDB")

In [23]:
# Bước 4 — Ingest cả 4 file
# Cả 2 file NVDA đều dùng symbol="NVDA" → where filter tự động lấy từ cả 2
ingest_company(NVDA_10Q,  "NVDA", collection)
ingest_company(NVDA_PRES, "NVDA", collection)
ingest_company(AMD_10Q,   "AMD",  collection)
ingest_company(AMD_PRES,  "AMD",  collection)
print(f"\nTổng số chunks trong collection: {collection.count()}")

[NVDA] 49 chunks từ NVDA-F1Q26-10-Q.pdf
[NVDA] Đã ingest 49 chunks vào ChromaDB
[NVDA] 9 chunks từ NVDA-F1Q26-Quarterly-Presentation-FINAL.pdf
[NVDA] Đã ingest 9 chunks vào ChromaDB
[AMD] 74 chunks từ AMD_F1Q26-10-Q.pdf
[AMD] Đã ingest 74 chunks vào ChromaDB
[AMD] 8 chunks từ AMD Q1'26 Earnings Slides Final.pdf
[AMD] Đã ingest 8 chunks vào ChromaDB

Tổng số chunks trong collection: 140


In [24]:
# Bước 5 — rag_agent(question, symbol)
# Khác với phase2: có thêm tham số symbol, dùng where filter khi query

def rag_agent(question, symbol):
    query_embedding = embed_model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
        # TODO: thêm where filter để chỉ lấy chunks của đúng symbol
        where = {"symbol": symbol}
    )

    context = "\n\n".join(results["documents"][0])

    prompt = (
        "Dựa vào thông tin sau từ báo cáo tài chính:\n\n"
        f"{context}\n\n"
        f"Hãy trả lời câu hỏi: {question}\n"
        "Nếu không đủ thông tin, hãy nói rõ."
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt
    )
    return response.text

In [25]:
# Bước 6 — Test 3 câu hỏi

print("=" * 60)
print("CÂU HỎI 1: NVIDIA")
print("=" * 60)
print(rag_agent("Doanh thu quý 1 năm 2026 là bao nhiêu?", "NVDA"))

print()
print("=" * 60)
print("CÂU HỎI 2: AMD")
print("=" * 60)
print(rag_agent("Doanh thu quý 1 năm 2026 là bao nhiêu?", "AMD"))

print()
print("=" * 60)
print("CÂU HỎI 3: So sánh cả 2 — dùng $in filter")
print("=" * 60)
query_vec = embed_model.encode(["quarterly revenue financial results"]).tolist()
r_both = collection.query(
    query_embeddings=query_vec,
    n_results=4,
    where={"symbol": {"$in": ["NVDA", "AMD"]}}
)
print("Top 4 chunks từ cả 2 công ty:")
for doc, meta in zip(r_both["documents"][0], r_both["metadatas"][0]):
    print(f"  [{meta['symbol']}] {doc[:80]}...")

CÂU HỎI 1: NVIDIA
Dựa trên thông tin được cung cấp, **không có đủ thông tin để xác định chính xác doanh thu quý 1 năm 2026**.

Các thông tin liên quan đến doanh thu hoặc thu nhập trong báo cáo này chủ yếu là về các khoản **chi phí phát sinh** hoặc **thay đổi trong tài sản/nợ phải trả**, chứ không trực tiếp đề cập đến tổng doanh thu.

Cụ thể, các điểm sau đây cho thấy thiếu thông tin:

*   **Amortization expense:** Báo cáo có đề cập đến chi phí khấu hao tài sản vô hình là 159 triệu đô la cho quý 1 năm tài chính 2026. Tuy nhiên, đây là chi phí, không phải doanh thu.
*   **Inventory provision:** Có đề cập đến khoản dự phòng hàng tồn kho cho quý 1 năm tài chính 2026 là 2,3 tỷ đô la. Đây là một khoản chi phí hoặc điều chỉnh giảm giá trị hàng tồn kho.
*   **Property and Equipment, Other Assets:** Các phần này liệt kê các khoản mục tài sản, không liên quan trực tiếp đến doanh thu.

Để biết doanh thu quý 1 năm 2026, cần phải tìm đến phần "Consolidated Statements of Operations" hoặc "Statements

In [26]:
# Bước 7 — Kiểm tra không bị data leakage
# Nếu where filter hoạt động đúng, query NVDA không bao giờ trả về chunk AMD

print("Kiểm tra: query NVDA không được trả về chunk AMD")
query_vec = embed_model.encode(["quarterly revenue financial results"]).tolist()
results = collection.query(
    query_embeddings=query_vec,
    n_results=5,
    where={"symbol": "NVDA"}
)

leakage_found = False
for i, meta in enumerate(results["metadatas"][0]):
    if meta["symbol"] != "NVDA":
        print(f"LEAKAGE tìm thấy ở chunk {i}: symbol={meta['symbol']}")
        leakage_found = True

if not leakage_found:
    print("OK — Tất cả chunks đều là NVDA, không bị lẫn dữ liệu AMD")
    print(f"Các chunks retrieved: {[m['symbol'] for m in results['metadatas'][0]]}")

Kiểm tra: query NVDA không được trả về chunk AMD
OK — Tất cả chunks đều là NVDA, không bị lẫn dữ liệu AMD
Các chunks retrieved: ['NVDA', 'NVDA', 'NVDA', 'NVDA', 'NVDA']
